# Module 2: Epidemic Modeling Template

## Team Members:
Delaney (Laney) Broderick
Conner Looney

## Project Title:
*(Fill in)*

## Project Goal:
This project seeks to estimate the initial growth rate from early outbreak data, predict the spread of the disease and when the cases will peak using SEIR modeling, make a guess for the family of the virus, and recommend a best intervention strategy to reduce the spread of the disease. 

## 1. Data and disease background
You can fill out this section throughout the module as you uncover more information about the mystery disease.

By the end of the module (when submitting), you should have some information about each of the following points:
* Prevalence & incidence in the UVA population (17900 undergraduate students)
*symptomatic period: 5-9 days
* Economic burden (you can generalize from respiratory viruses)
* Symptoms: mild-respiratory symptoms(predominate symptom), rash, low-grade fever, sore throat, fatigue, gastrointestinal symptoms
* Biological mechanisms (anatomy, organ physiology, cell & molecular physiology - you can generalize from viral biology)
*Transmission via respiratory droplets, pre-symptomatic transmission


## 2. Data Analysis
This section should be filled out sequentially as a full report of the work you've done over this module. You can copy and paste code from any main.py file here, and run it to produce plots. Once you gain more information throughout the module, you do not need to go back and "fix" earlier results. In other words, if your early predictions are found to be wrong when gaining new data, do not go back and rewrite them.

### 2a. Methods
#### Step 1: In python, we created a class of "virus_count" objects.


In [ ]:
class Virus_count : 
    all_virus_count = []

#### Step 2. We defined variables to describe the characteristics of the objects based on the data set, pulling the data from the .csv file of daily active case counts. 

In [ ]:
import csv
def __init__(self, day: int, date: str, active_reported_daily_cases : int): 
        self.day = day  
        self.date = date
        self.active_reported_daily_cases = active_reported_daily_cases

        Virus_count.all_virus_count.append(self) 

#### Step 3. Creating a representer to define what is printed from "virus_count" type object. Creating a getter method to get active reported daily cases and day. 

In [ ]:
def __repr__(self): 
        return f"( {self.day} | {self.date} | {self.active_reported_daily_cases})" 
    
def get_day(self): 
        return self.day

def get_active_reported_daily_cases(self): 
        return self.active_reported_daily_cases

#### Step 4. Creating 4 class methods. One to instantiate virus_count objects from a csv file, one to get a virus count object based on the day, one to get a virus count object based on daily active cases, one to filter virus_count objects based on the attributes of the virus_count class.


In [ ]:
@classmethod #creating a class method to instantiate virus_count objects from a csv file
def instantiate_from_csv(cls, filename: str):
        with open(filename, encoding="utf8") as f:
            reader = csv.DictReader(f)
            rows_of_virus_days = list(reader)
        for row in rows_of_virus_days:
            #creating a try except block to fix any value errors that may occur when trying to create virus_count objects from the csv file, if there is a value error it will just skip that row and move on to the next one
            try:    
                Virus_count(
                day = int(row['day']),
                date = row['date'],
                active_reported_daily_cases = int(row['active reported daily cases'])
            )
            except ValueError:
                continue

@classmethod #creating a class method to get a virus_count object based on the day of the virus_count (what we are interested in)
def get_virus_count_by_day(cls, day):
        for virus_count in Virus_count.all_virus_count:
            if day == virus_count.day: 
                return virus_count
            

@classmethod #creating a class method to get a virus_count object based on the active reported daily cases of the virus_count (what we are interested in)
def get_virus_count_by_active_cases(cls, active_reported_daily_cases):
        for virus_count in Virus_count.all_virus_count:
            if active_reported_daily_cases == virus_count.active_reported_daily_cases:
                return virus_count
            
@classmethod #creating a class method filter to filter virus_count objects based on the attributes of the virus_count class
def filter(cls, day: int = None, active_reported_daily_cases: int = None):
        all_virus_count = cls.all_virus_count
        remove_list = []
        attr_list = (
                        day,    
                        active_reported_daily_cases,
                        )
        attr_name = (
                        "day",
                        "active_reported_daily_cases",
                        )
        for attr in range(len(attr_list)):
            if attr_list[attr] is not None:
                for virus_count in all_virus_count:
                    if getattr(virus_count,attr_name[attr]) != attr_list[attr]:
                        remove_list.append(virus_count)
                all_virus_count = [virus_count for virus_count in all_virus_count if virus_count not in remove_list]
                remove_list.clear()
        return all_virus_count
    
        

*End code for data_release_1.py*_
*Start code for data_release_1_main.py*

#### Step 5. Importing data_release_1 class and the necessary libraries for the analysis of the virus data 

In [ ]:
from data_release_1 import *
import csv 
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import numpy as np
from scipy import stats
import statistics
import pandas as pd 

#### Step 6. Open csv file and instantiate from csv 

In [ ]:
with open(r"C:\Users\dance\OneDrive - University of Virginia\Computational BME\Module-2-Epidemics-SIR-Modeling-Broderick_Looney\Data\mystery_virus_daily_active_counts_RELEASE#1.csv", newline="") as f: #opening the csv file to get the headers of the csv file
    reader = csv.reader(f)
    headers = next(reader) # Get the first row to get the headers of the csv
    for h in headers:
       print(h) #show the headers of the csv
       
Virus_count.instantiate_from_csv(r"C:\Users\dance\OneDrive - University of Virginia\Computational BME\Module-2-Epidemics-SIR-Modeling-Broderick_Looney\Data\mystery_virus_daily_active_counts_RELEASE#1.csv")

#### Step 7. Create empty lists to store each virus_count object, and iterating through them

In [ ]:
day = []
active_reported_daily_cases = []

for virus_count in Virus_count.all_virus_count: 
    day.append(float(getattr(virus_count, "day")))
    active_reported_daily_cases.append(float(getattr(virus_count, "active_reported_daily_cases")))  



<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #1</b> 

</div>

 **What did you notice about the initial infections?**

The initial infections started off very slowly then increased at what looked to be an exponential growth model.

**How could we measure how quickly it's spreading?**

We could model the growth as an exponential function and then get an R0 value using that function and the infectious period (7-11 days so used average of 9 days).

**What information about the virus would be helpful in determining the shape of the outbreak curve?**

It would be helpful to know the lengths of the incubation, infectious, and recovery periods to better predict how the virus would interact in a community. It would also be helpful to know the means of infection.



### 2b. Plot the data & estimate initial growth rate (R0) from early data (through day 45)
This section should come from your python code after Data Release #1.

#### Step 8. Define the exponential function to fit the data

In [ ]:
def exponential_func(x, I0, r):
    return I0 * np.exp(r * x)

popt, pcov = curve_fit(exponential_func, X, y, p0=(1, 0.1))  # Initial guess for parameters a and b
print("I0,:", popt[0], "r:", popt[1])


#### Step 9. Calculate R0 using the two different ways we went over in class

In [ ]:
# Calulate R0 v1
R0_v1 = 1 + popt[1] * 9 # Assuming an infectious period of 9 days
print("R0_v1:", R0_v1)

#Calculate R0 v2
g = np.exp(popt[1])
R0_v2 = np.power(g, 9)
print("R0_v2:", R0_v2)

#### Step 10. Average the two R0 values to get our estimate of R0

In [ ]:
R0 = (R0_v1 + R0_v2) / 2
print("R0:", R0)

#### Step 11. Create a scatter plot to plot the Active Reported Daily Cases vs Day, and add exponential line of best fit

In [ ]:
plt.scatter(X, y, color='blue', label='All days')
plt.plot(X, exponential_func(X, *popt), color='red', label='Exponential fit')
plt.ylabel('Active Reported Daily Cases')
plt.xlabel('Day')
plt.title('Scatter Plot of Day vs Active Reported Daily Cases')
plt.legend()
plt.show()


**R0 estimate:**

Using two different methods our R0 value was 2.51

**Viruses with similar R0 values:**

1. West Nile Virus R0=2.5
    A mosquito-borne virus first identified in Uganda in 1937. 80% you get the virus are asymptomatic, while the other 20% experience mild flu-like symptoms. In severe cases the infection progresses to affect the central nervous system causing encephalitis or meningitis.
2. Ebola R0=2
    The Ebola virus is a bodily fluid transmitted virus most prevalent in Africa. It is severe and often fatal. It causes a viral hemorrhagic fever, meaning it can cause internal bleeding. The symptoms typically start with flu-like symptoms before the severe bleeding symptoms
3. Zika R0=3
    Typically transmitted by Aedes mosquitoes and was first identified in Uganda, although there was an outbreak in the US from 2015-2016. Many do not experience symptoms, but the common ones are fever, rash and joint pain. Zika virus during pregnancy can lead to serious congenital conditions like microcephaly and Guillain-Barre syndrome. 

**How accurate is your R0?**

R0_v1 = 2.078
R0_v2 = 2.94
Since our R0 value is an average of the two different methods to find R0, it is likely more accurate than just using one of the methods on its own. 

However with there being a pretty significant different in the two R0 values (0.862) there is some concern with the accuracy of the R0 value. 


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #2</b> 

</div>



### 2c. Use Euler's method to solve the SEIR model.
This section should come from your python code after Data Release #2.

In [ ]:
#Euler's method to find SIER model parameters
beta = 0.277    # transmission rate
sigma = 0.205    # incubation rate
gamma = 0.111    # recovery rate
h = 1          # time step
N = 17900    # total population

S0 = 17899
E0 = 0
I0 = 1
R0 = 0
day = 70
N = 17900
sigma = 0.205
beta = 0.277
gamma = 1/9
def seir(day,beta,sigma,gamma,S0,E0,I0,R0,N,h):
    S_list = []
    E_list = []
    I_list = []
    R_list = []
    
    S_list.append(S0)
    E_list.append(E0)
    I_list.append(I0)
    R_list.append(R0)

    for virus_day in range(day):
        h = 1
        # Euler derivatives
        dS = -beta * S_list[virus_day] * I_list[virus_day] / N
        dE = beta * S_list[virus_day] * I_list[virus_day] / N - sigma * E_list[virus_day]
        dI = sigma * E_list[virus_day] - gamma * I_list[virus_day]
        dR = gamma * I_list[virus_day]

        # Euler updates
        S_list.append(S_list[virus_day] + h * dS)
        E_list.append(E_list[virus_day] + h * dE)
        I_list.append(I_list[virus_day] + h * dI)
        R_list.append(R_list[virus_day] + h * dR)

    return S_list, E_list, I_list, R_list


### 2d. Fit the SEIR model to the data by changing beta, gamma, and sigma.
This section should come from your python code after Data Release #2.

Used Parameters:
    Beta = 0.277
    Sigma = 0.205
    Gamma = 0.111

Input Parameters for Grid Search:
    Beta = 0.3 (R0 * gamma)
    Sigma = 0.205
    Gamma = 0.111 
With a sigma value of 0.205 (incubation period of 4.1 days) grid search improved the SSE by 250047.19 (99.6% reduction)

Optimum Parameters:
    Beta = 0.357
    Sigma = 0.243
    Gamma = 0.107
SSE = 895.74

In [ ]:

data = pd.read_csv("/Users/connerlooney/Documents/GitHub/Module-2-Epidemics-SIR-Modeling-Broderick_Looney/Data/mystery_virus_daily_active_counts_RELEASE#2.csv")
#data = pd.read_csv(r"C:\Users\dance\OneDrive - University of Virginia\Computational BME\Module-2-Epidemics-SIR-Modeling-Broderick_Looney\Data\mystery_virus_daily_active_counts_RELEASE#2.csv")
data = data["active reported daily cases"].tolist()

def grid_search(day,N,S0,E0,I0,R0,data):
    beta = np.linspace(0.2, 0.7)
    sigma = np.linspace(0.1, 0.3)
    gamma = np.linspace(0.05, 0.25)
    SSE = []
    beta_list = []
    sigma_list = []
    gamma_list = []
    
    best_SSE = float("inf")
    best_beta = None
    best_sigma = None
    best_gamma = None

    # iterate through parameter combinations
    for b in beta_values:
        for s in sigma_values:
            for g in gamma_values:
                # run Euler method for these parameters
                S_list, E_list, I_list, R_list = seir(day, b, s, g, S0, E0, I0, R0, N, h)

                model_I = I_list[: len(data)]

                # compute sum of squared errors
                errors = [(model_I[i] - data[i]) ** 2 for i in range(len(model_I))]
                sse = sum(errors)

                # record results
                SSE_list.append(sse)
                beta_list.append(b)
                sigma_list.append(s)
                gamma_list.append(g)

                # update best if this is lowest SSE so far
                if sse < best_SSE:
                    best_SSE = sse
                    best_beta = b
                    best_sigma = s
                    best_gamma = g
    
    return best_beta, best_sigma, best_gamma, best_SSE, SSE_list, beta_list, sigma_list, gamma_list

# perform grid search using defined parameters
best_beta, best_sigma, best_gamma, best_SSE, SSE_list, beta_list, sigma_list, gamma_list = \
    grid_search(day, N, S0, E0, I0, R0, data)

# print out best parameters and associated SSE
print(f"Best parameters from grid search:\n  beta = {best_beta}\n  sigma = {best_sigma}\n  gamma = {best_gamma}\n  SSE = {best_SSE}")

# generate model output with optimal parameters
S_list_opt, E_list_opt, I_list_opt, R_list_opt = seir(day, best_beta, best_sigma, best_gamma, S0, E0, I0, R0, N, h=1)




### 2e. Plot the model-predicted infections over time compared to the data.
This section should come from your python code after Data Release #2.

From the code: 
Best parameters from grid search:
  beta = 0.5265306122448979
  sigma = 0.28775510204081634
  gamma = 0.20510204081632655
  SSE = 97574.40406130366

In [ ]:
plt.plot(range(day+1), S_list_opt, label="Susceptible")
plt.plot(range(day+1), E_list_opt, label="Exposed")
plt.plot(range(day+1), I_list_opt, label="Infectious")
plt.plot(range(day+1), R_list_opt, label="Recovered")

plt.xlabel("Day")
plt.ylabel("Population")
plt.title("SEIR Model (best fit)")
plt.legend()

plt.show()

### 2e. Predict the day and amount of active cases at the peak of the epidemic spread.
This section should come from your python code after Data Release #2.


In [ ]:
extended_days = 200  # run model well past the observed 70 days
S_ext, E_ext, I_ext, R_ext = seir(extended_days, best_beta, best_sigma, best_gamma, S0, E0, I0, R0, N, h=1)

peak_I = max(I_ext)
peak_day = I_ext.index(peak_I)
print(f"Extended simulation peak infectious count: {peak_I:.2f} on day {peak_day}")



**Predicted Peak**
Extended simulation peak infectious count: 2609.94 on day 73


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #3</b> 

</div>



### 2f. Plot the full dataset (Data Release #3) against your model.
This section should come from your python code after Data Release #3.


***Insert csv of data release #3***

In [ ]:
data_3 = pd.read_csv(r"C:\Users\dance\OneDrive - University of Virginia\Computational BME\Module-2-Epidemics-SIR-Modeling-Broderick_Looney\Data\mystery_virus_daily_active_counts_RELEASE#3.csv")
data_3 = data_3["active reported daily cases"].tolist()


**compare the full release #3 dataset against the SEIR model using best parameters**

In [ ]:
model_S3, model_E3, model_I3, model_R3 = seir(len(data_3), best_beta, best_sigma, best_gamma, S0, E0, I0, R0, N, h=1)

plt.figure()
plt.plot(range(len(data_3)), model_I3[:len(data_3)], label="Model Infectious")
plt.scatter(range(len(data_3)), data_3, color='red', label="Observed active cases")
plt.xlabel("Day")
plt.ylabel("Infectious / Reported Active Cases")
plt.title("Model vs Data (Release #3)")
plt.legend()
plt.show()

### 2g. Intervention strategies for new outbreak at VT (70 days of infection)
This section should come from your python code after Data Release #3.



## Verify and validate your analysis: 

*(Describe how you checked to see that your analysis gave you an answer that you believe (verify). Describe how your determined if your analysis gave you an answer that is supported by other evidence (e.g., a published paper).*

## Conclusions and Ethical Implications: 
*(Think about the answer your analysis generated, draw conclusions related to your overarching question, and discuss the ethical implications of your conclusions.*

## Limitations and Future Work: 
*(Think about the answer your analysis generated, draw conclusions related to your overarching question, and discuss the ethical implications of your conclusions.*